In [58]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [59]:
# analytic calculation of hitting time by formula
# h₀ = Σⱼ₌₀ᴺ⁻¹ [1/(λⱼ·πⱼ) · Σₖ₌₀ʲ πₖ] 
# where πₖ = (λ₀λ₁...λₖ₋₁)/(μ₁μ₂...μₖ) for k ≥ 1, and π₀ = 1

from typing import Sequence

def hitting_time(fs: Sequence[float], rs: Sequence[float]) -> float:
    """
    Compute the expected hitting time from state 0 to state N in a birth-death process
    with birth rates fs and death rates rs.

    fs: sequence of birth rates λ₀, λ₁, ..., λₙ₋₁
    rs: sequence of death rates μ₁, μ₂, ..., μₙ

    Returns the expected hitting time h0.
    """
    assert len(fs) == len(rs), "fs and rs must have the same length"
    N = len(fs)  # Number of states is N+1 (0 to N)

    # Step 1: Compute πₖ = (λ₀λ₁...λₖ₋₁)/(μ₁μ₂...μₖ) for k ≥ 1, and π₀ = 1
    pis = [0.0] * (N + 1)
    pis[0] = 1.0
    for k in range(1, N + 1):
        pis[k] = pis[k-1] * fs[k - 1] / rs[k - 1]

    # Step 2: Compute h_0 using the formula
    # h₀ = Σⱼ₌₀ᴺ⁻¹ [1/(λⱼ·πⱼ) · Σₖ₌₀ʲ πₖ]
    h_0 = 0.0
    sum_pi = 0.0
    for j in range(N):
        sum_pi += pis[j]
        h_0 += (1 / (fs[j] * pis[j])) * sum_pi

    return h_0

In [60]:
%%cython --cplus

# numerical simulation of hitting time using Cython

import math
from typing import Sequence
import cython
import numpy as np
from tqdm.auto import tqdm

# Import C math functions for fast random number generation
from libc.math cimport log
from libc.stdlib cimport rand, RAND_MAX, srand
from libc.time cimport time
from libcpp.vector cimport vector # C++ vector

# Fast Cython version using C library
@cython.boundscheck(False)
@cython.wraparound(False)
def simulate_hitting_time_cython(
    fs: Sequence[float], rs: Sequence[float], trials: int, progress: bool = False
) -> float:
    """
    Fast Cython simulation using C library random numbers
    """
    assert len(fs) == len(rs), "fs and rs must have the same length"
    cdef int N = len(fs)
    cdef int trial, state
    cdef double time_val, birth_rate, death_rate, total_rate, rand_val, event_time
    
    # Pre-convert to numpy arrays
    cdef double[:] fs_array = np.array(fs, dtype=np.float64)
    cdef double[:] rs_array = np.array(rs, dtype=np.float64)
    cdef double[:] hitting_times = np.empty(trials, dtype=np.float64)

    # C++ vector for step times
    cdef vector[double] step_times
    
    # Seed C random number generator
    srand(<unsigned int>time(NULL))

    iterator = tqdm(range(trials), desc="Fast Cython simulation") if progress else range(trials)
    for trial in iterator:
        state = 0

        step_times.clear()
        while state < N:
            # Calculate total rate for current state
            birth_rate = fs_array[state] if state < N else 0.0
            death_rate = rs_array[state - 1] if state > 0 else 0.0
            total_rate = birth_rate + death_rate
            
            if total_rate > 0:
                # Generate exponential random variable using inverse transform
                # -log(U) / rate where U is uniform(0,1)
                rand_val = <double>rand() / <double>RAND_MAX
                if rand_val == 0.0:  # Avoid log(0)
                    rand_val = 1e-15
                event_time = -log(rand_val) / total_rate
                step_times.push_back(event_time)

                # Determine which event occurs (birth vs death)
                rand_val = <double>rand() / <double>RAND_MAX
                if rand_val < birth_rate / total_rate:
                    state += 1  # Birth
                else:
                    state -= 1  # Death
            else:
                break  # No valid transitions
            
        time_val = sum(step_times)
        hitting_times[trial] = time_val

    return float(np.mean(hitting_times))

In [24]:
fs = [1,2,3,4,5,6,3,1,7,6,5,3,3,2,1,1,2,3,4,5,6,3,1,7,6,5,3,3,2,1]*2
rs =   [2,3,4,3,6,7,1,3,6,5,4,3,2,1,1,2,3,4,3,6,7,1,3,6,5,4,3,2,1,1]*2

%timeit hitting_time(fs, rs)
%timeit simulate_hitting_time_cython(fs, rs, 1)

8.08 μs ± 408 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
129 μs ± 41.9 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [33]:
# fs = [1,2,3,4,5,6,3,1,7,6,5,3,3,2,1,1,2,3,4,5,6,3,1,7,6,5,3,3,2,1]*2
# rs =   [2,3,4,3,6,7,1,3,6,5,4,3,2,1,1,2,3,4,3,6,7,1,3,6,5,4,3,2,1,1]*2
fs = [1,2,3,4,5,6,3,1,7,6,5,3,3,2,1]
rs =   [2,3,4,3,6,7,1,3,6,5,4,3,2,1,1]
trials = 10**6

print("Analytic hitting time:", hitting_time(fs, rs))
# print("Simulated hitting time:", simulate_hitting_time(fs, rs, trials))
print("Simulated hitting time Cython:", simulate_hitting_time_cython(fs, rs, trials, progress=True))

Analytic hitting time: 44.94126984126984


Fast Cython simulation:   0%|          | 0/1000000 [00:00<?, ?it/s]

Simulated hitting time Cython: 44.783646073117595


In [ ]:
import math
import numpy as np
import nupack
import nuad.vienna_nupack as nv

model=nupack.Model(material='dna')
toehold = 'GTGGGT'
bm_domain = 'ACCGCACCACGTGGGTGTCG'
invader = toehold + bm_domain
bot_strand = nv.rc(invader)
incumbent = bm_domain

structs = [ # initial state: all of toehold but last base is paired
    '(' * (len(toehold) - 1) + '.' * (len(bm_domain) + 1) + 
    '+' + '(' * len(bm_domain) +
    '+' + ')' * len(bm_domain) + '.' + ')' * (len(toehold) - 1)
]
# print(f'initial structure: {structs[0]}')
for i in range(len(bm_domain)):
    struct_inv = '(' * (len(toehold) + i) + '.' * (len(bm_domain) - i)
    struct_inc = '.' * i + '(' * (len(bm_domain) - i)
    struct_bot = ')' * (len(toehold) + len(bm_domain))
    struct = struct_inv + '+' + struct_inc + '+' + struct_bot
    structs.append(struct)
    # print(struct)
    struct_inc_break_one_bp_inc = '.' * (i+1) + '(' * (len(bm_domain) - i - 1)
    struct_bot_break_one_bp_inc = ')' * (len(bm_domain) - i - 1) + '.' + ')' * (len(toehold) + i)
    struct_break_one_bp_inc = struct_inv + '+' + struct_inc_break_one_bp_inc + '+' + struct_bot_break_one_bp_inc
    structs.append(struct_break_one_bp_inc)
    # print(struct_break_one_bp_inc)
    # print()

Gp = -1.2

from pprint import pprint
# pprint(structs)
energies = []
for i, struct in enumerate(structs):
    energy = nupack.structure_energy((invader, incumbent, bot_strand), struct, model)
    # print(f'energy: {energy}')
    if i == 1:
        energy += Gp # add penalty for initial state
        # print(f'energy after Gp: {energy}')
    # print(f'free energy of structure {struct}: {energy}')
    energies.append(energy)

fs = [] # fs[i] = forward rate from state i to i+1
rs = [] # rs[i] = reverse rate from state i+1 to i
# so transition rates out of state i are fs[i] and rs[i-1]
# fs not defined for i=N, rs not defined for i=0

R = 0.0019872041 # kcal/mol/K
T = 37 + 273.15  # K
kuni = 7.3e6     # ??? no idea what this should be
from itertools import pairwise
diffs = []
for e1, e2 in pairwise(energies):
    diffs.append(abs(e2 - e1))
    if e1 > e2:
        fs.append(kuni)
        rs.append(kuni * math.exp((e1 - e2) / (R * T)))
    else:
        fs.append(kuni * math.exp((e2 - e1) / (R * T)))
        rs.append(kuni)

print(f'mean diff: {np.mean(diffs)}')
# print(f'{fs=}')
# print(f'{rs=}')
print(f'analytic hitting time: {hitting_time(fs, rs) * 1e3:.3f} ms')
# print(f'sampled hitting time: {simulate_hitting_time_cython(fs, rs, 1000, progress=True) * 1e6} us')

mean diff: 1.6266535312200587
analytic hitting time: 0.290 ms
